In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor,HistGradientBoostingRegressor
import xgboost as xgb
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
train.head()

,id,emotional_charge_2,groove_efficiency_1,beat_frequency_1,organic_texture_2,composition_label_0,harmonic_scale_1,intensity_index_0,duration_ms_0,album_name_length,...,time_signature_0,duration_ms_1,harmonic_scale_0,time_signature_2,rhythmic_cohesion_2,emotional_resonance_0,harmonic_scale_2,intensity_index_2,instrumental_density_0,target
0,76339,0.482850,1.169231,80.018,0.0201,Country Stuff (feat. Jake Owen),1.0,0.789,154586.0,NaN,...,4.0,161853.0,7.0,4.0,NaN,0.607,7.0,0.7250,0.000000,74
1,80006,0.267862,1.321321,147.966,0.3340,Solitude,6.0,0.715,46874.0,15.0,...,4.0,155619.0,1.0,4.0,0.843,0.783,4.0,NaN,0.043200,2
2,83501,0.242606,1.285319,142.980,0.1110,BDFFRNT (Saved from Conformity),4.0,NaN,264665.0,7.0,...,4.0,209378.0,6.0,4.0,NaN,0.211,10.0,0.6020,0.000000,35
3,81530,0.426400,1.279435,123.063,0.1960,Headlights (feat. Ilsey),5.0,0.685,209208.0,5.0,...,4.0,219043.0,11.0,4.0,0.702,0.369,NaN,0.8200,0.000335,70
4,60534,0.000000,0.974906,132.722,0.0811,Afraid,6.0,0.856,215346.0,5.0,...,4.0,258893.0,1.0,0.0,0.000,0.631,1.0,0.0221,0.000000,78


In [4]:
test.head()
test_ids = test["id"]

In [5]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 61609 entries, 0 to 61608
Data columns (total 62 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          61609 non-null  int64  
 1   emotional_charge_2          59167 non-null  float64
 2   groove_efficiency_1         61429 non-null  float64
 3   beat_frequency_1            61223 non-null  float64
 4   organic_texture_2           61226 non-null  float64
 5   composition_label_0         58660 non-null  str    
 6   harmonic_scale_1            58304 non-null  float64
 7   intensity_index_0           55638 non-null  float64
 8   duration_ms_0               60320 non-null  float64
 9   album_name_length           52015 non-null  float64
 10  beat_frequency_0            51878 non-null  float64
 11  beat_frequency_2            59843 non-null  float64
 12  artist_count                58348 non-null  float64
 13  composition_label_1         60149 non-null

In [6]:
train.describe()

,id,emotional_charge_2,groove_efficiency_1,beat_frequency_1,organic_texture_2,harmonic_scale_1,intensity_index_0,duration_ms_0,album_name_length,beat_frequency_0,...,time_signature_0,duration_ms_1,harmonic_scale_0,time_signature_2,rhythmic_cohesion_2,emotional_resonance_0,harmonic_scale_2,intensity_index_2,instrumental_density_0,target
count,61609.000000,59167.000000,61429.000000,61223.000000,61226.000000,58304.000000,55638.000000,6.032000e+04,52015.000000,51878.000000,...,59704.000000,5.250400e+04,53925.000000,58455.000000,56049.000000,60063.000000,57142.000000,60916.000000,60900.000000,61609.000000
mean,51390.780162,0.316976,1.238856,121.022910,0.274748,5.192594,0.604426,2.011315e+05,18.225723,119.133973,...,3.874849,2.110477e+05,5.212499,3.901274,0.612252,0.458851,5.288894,0.616045,0.148391,52.067328
std,29659.344472,0.212777,6.171617,30.467061,0.303020,3.629153,0.243943,1.100738e+05,14.404713,32.067971,...,0.564558,8.911099e+04,3.571288,0.465295,0.179591,0.261196,3.567118,0.230109,0.306915,21.569248
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.485000e+03,1.000000,0.000000,...,0.000000,4.120000e+03,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,25832.000000,0.143877,0.730914,96.138000,0.027125,2.000000,0.447000,1.482340e+05,9.000000,94.802250,...,4.000000,1.682000e+05,2.000000,4.000000,0.506000,0.243000,2.000000,0.477000,0.000000,37.000000
50%,51410.000000,0.291060,1.004894,120.012000,0.141000,5.000000,0.633000,1.959215e+05,14.000000,119.893000,...,4.000000,2.029730e+05,5.000000,4.000000,0.630000,0.449000,5.000000,0.646000,0.000019,57.000000
75%,77069.000000,0.466860,1.358251,141.401000,0.454000,8.000000,0.803000,2.402488e+05,23.000000,140.023000,...,4.000000,2.413605e+05,8.000000,4.000000,0.745000,0.663000,8.000000,0.791000,0.024600,69.000000
max,102681.000000,0.976063,654.000000,239.983000,0.996000,11.000000,1.000000,3.664274e+06,199.000000,235.998000,...,5.000000,3.550973e+06,11.000000,5.000000,0.979000,1.000000,11.000000,1.000000,1.000000,100.000000


In [7]:
train.dtypes.unique()

array([dtype('int64'), dtype('float64'),
       <StringDtype(storage='python', na_value=nan)>], dtype=object)

In [8]:
cat_cols = train.select_dtypes(include=["str" ]).columns
num_cols = train.select_dtypes(include=["int64", "float64"]).columns

unique_counts = train[cat_cols].nunique().sort_values(ascending=False)
print(unique_counts.head(10))

track_identifier         22991
composition_label_1      22930
composition_label_2      22169
composition_label_0      21709
creator_collective       15139
publication_timestamp     6061
weekday_of_release           7
season_of_release            4
lunar_phase                  4
dtype: int64


In [9]:
for df in [train, test]:
    df["publication_timestamp"] = pd.to_datetime(df["publication_timestamp"], errors="coerce")
    df["pub_year"] = df["publication_timestamp"].dt.year
    df["pub_month"] = df["publication_timestamp"].dt.month
    df["pub_day"] = df["publication_timestamp"].dt.day
    df["pub_dayofweek"] = df["publication_timestamp"].dt.dayofweek

train.drop(columns=["publication_timestamp"], inplace=True)
test.drop(columns=["publication_timestamp"], inplace=True)

In [10]:
y = train["target"]
X = train.drop(columns=["target"])
X_test = test.copy()

In [11]:
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size=0.2, random_state=42) 

In [12]:
drop_cols = ["id", "track_identifier","composition_label_0", "composition_label_1","composition_label_2","creator_collective"]

X_train = X_train.drop(columns=drop_cols)
X_val = X_val.drop(columns=drop_cols)
X_test_model = X_test.drop(columns=drop_cols)

In [13]:
num_cols = X_train.select_dtypes(["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(["str","object"]).columns


corrs = X_train[num_cols].corr()
corrs_abs = corrs.abs() 
high_corr = corrs_abs > 0.5
cols = corrs_abs.columns
result = [(cols[i], cols[j]) 
          for i in range(len(cols)) 
          for j in range(i) 
          if corrs_abs.iloc[i, j] > 0.5]

print(result)


[('emotional_charge_1', 'emotional_charge_2'), ('emotional_charge_0', 'intensity_index_0'), ('emotional_charge_0', 'emotional_charge_1'), ('duration_ms_2', 'duration_ms_0'), ('organic_texture_0', 'organic_texture_2'), ('organic_texture_0', 'intensity_index_0'), ('vocal_presence_1', 'vocal_presence_2'), ('vocal_presence_0', 'vocal_presence_2'), ('vocal_presence_0', 'vocal_presence_1'), ('intensity_index_1', 'organic_texture_2'), ('intensity_index_1', 'intensity_index_0'), ('intensity_index_1', 'emotional_charge_1'), ('organic_immersion_0', 'duration_ms_0'), ('organic_immersion_0', 'organic_texture_0'), ('instrumental_density_1', 'instrumental_density_2'), ('organic_immersion_2', 'organic_texture_2'), ('organic_immersion_2', 'duration_ms_2'), ('organic_immersion_2', 'organic_immersion_0'), ('organic_texture_1', 'organic_texture_2'), ('organic_texture_1', 'organic_texture_0'), ('organic_texture_1', 'intensity_index_1'), ('emotional_resonance_1', 'emotional_charge_1'), ('rhythmic_cohesion_

In [14]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy = "median")),
    
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ("num",num_pipeline, num_cols ),
    ("cat", cat_pipeline, cat_cols)
])

In [15]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=80,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=80,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),
    "DecisionTree": DecisionTreeRegressor(
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    ),
    "Gradient Boost": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        random_state=42
    ),
    "Ada Boost": AdaBoostRegressor(
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators = 20000,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
        ),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=300,
        min_samples_leaf=20,
        random_state=42
        )
}

In [16]:
results = []
trained_models = {}

for name, regressor in models.items():
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", regressor) 
    ])

    model.fit(X_train,y_train)
    val_preds = model.predict(X_val)
    
    
    # Model evaluation
    mse = mean_squared_error(y_val, val_preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_val, val_preds)
    r2 = r2_score(y_val, val_preds)


    results.append(
        {
            "Model": name,
            "MSE": mse,
            "RMSE": rmse,
            "MAE": mae,
            "R^2": r2
        }
    )

    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values("RMSE")
print(results_df)
    

                           Model         MSE       RMSE        MAE       R^2
7                        XGBoost  102.624733  10.130387   5.711322  0.779655
2                   RandomForest  171.273606  13.087154   9.358207  0.632260
8  HistGradientBoostingRegressor  193.463933  13.909131  10.357714  0.584615
3                     ExtraTrees  198.875404  14.102319  10.550190  0.572996
5                 Gradient Boost  260.663749  16.145084  12.751448  0.440331
4                   DecisionTree  262.933774  16.215233  11.114492  0.435457
6                      Ada Boost  369.937608  19.233762  15.895828  0.205710
0               LinearRegression  375.529957  19.378595  15.642679  0.193703
1                          Ridge  375.530024  19.378597  15.642875  0.193703


In [17]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

test_preds = best_model.predict(X_test_model)

In [18]:
submission = pd.DataFrame({
    "id": test_ids,
    "target": test_preds
})

submission.to_csv("submission.csv", index=False)
